# Synthetic end-to-end pipeline

Notebook 01 checks the environment and fixture. This notebook runs the synthetic corpus, lineage,
TF-IDF retrieval and role-time aggregation through their usual interfaces. It records each run in
`outputs/demo/runs` and verifies its manifest before showing tables. The fixture results are
interface tests on invented data.

To run the optional real model demo, provide the pinned local model snapshots and set
`FOT_RUN_OFFLINE_MODELS=1` before starting the kernel. A missing snapshot blocks that mode.


In [ ]:
import hashlib
import json
import os
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from tempfile import TemporaryDirectory
from uuid import uuid4

import duckdb
import pandas as pd

from fear_temperature.cleaning import VALID_ROLES, prepare_corpus
from fear_temperature.config import PROJECT_ROOT, Settings, load_yaml
from fear_temperature.corpus import store_corpus
from fear_temperature.io import read_records
from fear_temperature.retrieval import retrieval_fixture_summary, tfidf_retrieve
from fear_temperature.temporal import aggregate_role_time

config_path = PROJECT_ROOT / "configs/default.yaml"
models_path = PROJECT_ROOT / "configs/models.yaml"
config = load_yaml(config_path)
demo = config["demo"]
fixture_path = PROJECT_ROOT / "data/fixtures/synthetic_passages.jsonl"
expected_fixture = "9379653ca6b2240b96d8c3e4a53d0f23b72e4300199cd2a3737dae790ff068ce"


def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


if PROJECT_ROOT / demo["input"] != fixture_path or digest(fixture_path) != expected_fixture:
    raise ValueError("The synthetic fixture differs from the fixture checked in Notebook 01.")

input_digests = {
    p.relative_to(PROJECT_ROOT).as_posix(): digest(p)
    for p in (fixture_path, config_path, models_path)
}
runs = PROJECT_ROOT / "outputs/demo/runs"
runs.mkdir(parents=True, exist_ok=True)


def file_manifest(folder):
    files = {}
    for file in sorted(folder.rglob("*")):
        if not file.is_file() or file == folder / "manifest.json":
            continue
        if file.is_symlink():
            raise ValueError(f"Unexpected symlink in run: {file.name}")
        entry = {"bytes": file.stat().st_size, "sha256": digest(file)}
        if file.suffix == ".parquet":
            import pyarrow.parquet as pq

            info = pq.ParquetFile(file)
            entry.update(
                rows=info.metadata.num_rows,
                columns={f.name: str(f.type) for f in info.schema_arrow},
            )
        files[file.relative_to(folder).as_posix()] = entry
    return files


def validate_run(folder, expected_inputs):
    manifest = json.loads((folder / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("status") != "completed" or manifest["run_id"] != folder.name:
        raise ValueError("Run is incomplete or the manifest does not identify this run.")
    if manifest["inputs"] != expected_inputs:
        raise ValueError("Run inputs differ from the current fixture or configuration.")
    if manifest["files"] != file_manifest(folder):
        raise ValueError("Run output is missing, unexpected, or fails its manifest checks.")
    return manifest


def publish_run(folder, mode, details):
    current_inputs = {name: digest(PROJECT_ROOT / name) for name in input_digests}
    if current_inputs != input_digests:
        raise ValueError("A fixture or configuration changed during the run.")
    manifest = {
        "run_id": folder.name,
        "mode": mode,
        "status": "completed",
        "inputs": input_digests,
        "files": file_manifest(folder),
        **details,
    }
    (folder / "manifest.json").write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    validate_run(folder, input_digests)
    destination = runs / folder.name
    folder.rename(destination)
    return destination

In [ ]:
counts_expected = {"input": 37, "lineage": 37, "canonical_passages": 36, "duplicates": 1}
records = read_records(fixture_path)
corpus = prepare_corpus(records)
counts = {
    "input": len(records),
    "lineage": len(corpus.lineage),
    "canonical_passages": len(corpus.passages),
    "duplicates": int(corpus.lineage["is_exact_duplicate"].sum()),
}
if counts != counts_expected or set(corpus.passages["publisher_role"]) != VALID_ROLES:
    raise ValueError(f"Synthetic corpus interface check failed: {counts}")

passages = corpus.passages
retrieval = tfidf_retrieve(
    passages["text"].tolist(),
    passages["passage_id"].tolist(),
    demo["retrieval_query"],
    int(demo["semantic_top_k"]),
)
retrieval_summary = retrieval_fixture_summary(retrieval, passages)
retrieval_counts = (
    retrieval_summary[["retrieved", "fixture_relevant_at_k", "fixture_future_worry_at_k"]]
    .iloc[0]
    .tolist()
)
if (
    list(retrieval.columns) != ["method", "rank", "passage_id", "score"]
    or retrieval_counts != [10, 10, 6]
    or retrieval["rank"].tolist() != list(range(1, 11))
    or not retrieval["passage_id"].is_unique
):
    raise ValueError(f"Unexpected TF-IDF fixture interface: {retrieval_counts}")

aggregation = aggregate_role_time(
    passages,
    frequency=config["aggregation"]["frequency"],
    roles=tuple(config["aggregation"]["roles"]),
)
source_months = pd.to_datetime(passages["publication_date"]).dt.to_period("M")
expected_months = pd.period_range(source_months.min(), source_months.max(), freq="M")
expected_grid = {(month.to_timestamp(), role) for month in expected_months for role in VALID_ROLES}
observed_grid = set(aggregation[["time", "publisher_role"]].itertuples(index=False, name=None))
if (
    observed_grid != expected_grid
    or aggregation.duplicated(["time", "publisher_role"]).any()
    or aggregation["eligible_n"].sum() != counts["canonical_passages"]
    or not (aggregation["future_worry_n"] <= aggregation["relevant_n"]).all()
    or not (aggregation["relevant_n"] <= aggregation["eligible_n"]).all()
):
    raise ValueError("The synthetic role-time aggregation failed its interface checks.")

with TemporaryDirectory(prefix=".synthetic-", dir=runs) as temporary:
    work = Path(temporary) / ("fixture-" + uuid4().hex[:12])
    storage = store_corpus(corpus, work / "storage")
    with duckdb.connect(storage["database"], read_only=True) as connection:
        observed_rows = connection.execute(
            "SELECT count(*), count(*) FILTER (WHERE is_exact_duplicate) FROM passage_evidence"
        ).fetchone()
    if observed_rows != (37, 1):
        raise ValueError(f"DuckDB lineage check failed: {observed_rows}")
    tables = work / "tables"
    tables.mkdir()
    retrieval.to_parquet(tables / "retrieval.parquet", index=False)
    retrieval_summary.to_parquet(tables / "retrieval_fixture_summary.parquet", index=False)
    aggregation.to_parquet(tables / "aggregation_synthetic_expected.parquet", index=False)
    fixture_run = publish_run(
        work,
        "synthetic_fixture",
        {"counts": counts, "tfidf_counts": retrieval_counts, "role_time_rows": len(aggregation)},
    )

checked = validate_run(fixture_run, input_digests)
print(
    {
        "run": fixture_run.relative_to(PROJECT_ROOT).as_posix(),
        "counts": checked["counts"],
        "files_verified": len(checked["files"]),
    }
)
display(pd.read_parquet(fixture_run / "tables/retrieval_fixture_summary.parquet"))
display(pd.read_parquet(fixture_run / "tables/aggregation_synthetic_expected.parquet"))

In [ ]:
models = load_yaml(models_path)
settings = Settings.from_env()
needed = ("sentence_encoder", "emotion_candidate")
missing = []
for name in needed:
    entry = models[name]
    repository = f"models--{entry['id'].replace('/', '--')}"
    snapshot = settings.model_cache / repository / "snapshots" / entry["revision"]
    if not snapshot.is_dir():
        missing.append(name)
try:
    parser_installed = version(models["parser"]["id"])
except PackageNotFoundError:
    parser_installed = None
if parser_installed != models["parser"]["revision"]:
    missing.append("configured spaCy model version")


def verify_model_report(folder, report):
    if report["status"] != "completed":
        raise ValueError("Offline model run did not complete.")
    if {
        key: report["record_counts"][key] for key in ("raw", "canonical_passages", "duplicates")
    } != {
        "raw": 37,
        "canonical_passages": 36,
        "duplicates": 1,
    }:
        raise ValueError("Offline model run has unexpected fixture counts.")
    for name in needed:
        if report["models"][name]["loaded_snapshot_revision"] != models[name]["revision"]:
            raise ValueError(f"Loaded snapshot revision differs for {name}.")
    if report["models"]["parser"]["loaded_version"] != models["parser"]["revision"]:
        raise ValueError("Loaded spaCy model version differs.")
    expected = file_manifest(folder)
    observed = {}
    for entry in report["outputs"]:
        file = (PROJECT_ROOT / entry["path"]).resolve()
        relative = file.relative_to(folder.resolve()).as_posix()
        observed[relative] = {key: entry[key] for key in ("bytes", "sha256")}
    if observed != {
        name: {k: row[k] for k in ("bytes", "sha256")} for name, row in expected.items()
    }:
        raise ValueError("Offline model run output ledger does not match the staged files.")


if os.environ.get("FOT_RUN_OFFLINE_MODELS") == "1":
    if missing:
        raise RuntimeError(f"Offline model run unavailable. Missing: {', '.join(missing)}.")
    from fear_temperature.pipeline import run_demo

    with TemporaryDirectory(prefix=".offline-models-", dir=runs) as temporary:
        temp = Path(temporary)
        work = temp / ("offline-models-" + uuid4().hex[:12])
        run_config = {**config, "demo": {**demo, "output": str(work)}}
        config_copy = temp / "local.yaml"
        config_copy.write_text(json.dumps(run_config), encoding="utf-8")
        report = run_demo(config_path=config_copy, local_files_only=True)
        verify_model_report(work, report)
        model_run = publish_run(
            work,
            "offline_models",
            {
                "counts": report["record_counts"],
                "models": report["models"],
                "device_probe": report["device_probe"],
                "packages": report["packages"],
            },
        )
    print(
        {
            "run": model_run.relative_to(PROJECT_ROOT).as_posix(),
            "models": validate_run(model_run, input_digests)["models"],
        }
    )
else:
    print({"offline_model_demo": "SKIPPED", "missing_prerequisites": missing})